In [1]:
import os
from pathlib import Path

# PySpark 4.x needs Java 17+; shell may still set JAVA_HOME to Java 11.
for _jdk in (Path("/opt/homebrew/opt/openjdk@17"), Path("/usr/local/opt/openjdk@17")):
    if (_jdk / "bin" / "java").is_file():
        os.environ["JAVA_HOME"] = str(_jdk.resolve())
        break

# Avoid slow / flaky bind when the Mac hostname maps to loopback (Spark WARNs about this).
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("Ecommerce Pipeline")
    .master("local[1]")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark started (first run can take ~30–60s while the JVM starts; later cells are much faster)")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/07 16:59:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark started (first run can take ~30–60s while the JVM starts; later cells are much faster)


In [2]:
from pathlib import Path


def sales_csv_path() -> str:
    for p in (Path("sales.csv"), Path("day2_de_vector_dbs") / "sales.csv"):
        if p.is_file():
            return str(p.resolve())
    raise FileNotFoundError(
        "sales.csv not found. Run the notebook with project root (uptime) or day2_de_vector_dbs as cwd."
    )


input_path = sales_csv_path()

In [ ]:
Spark== Lazy Evaluation 


In [4]:
df = spark.read.csv(input_path, header=True, inferSchema=True)

In [5]:
df.show()

+--------+-----------+--------------+----------+--------+---------------+------------+----------+
|order_id|customer_id|transaction_id|product_id|quantity|discount_amount|total_amount|order_date|
+--------+-----------+--------------+----------+--------+---------------+------------+----------+
|       1|          1|          1001|         3|       3|           9.02|     1300.12|2023-12-02|
|       2|          5|          1002|         4|       3|          17.74|      353.99|2023-12-04|
|       3|          1|          1003|         5|       2|          14.78|      389.16|2023-12-05|
|       4|          5|          1004|         5|       3|          11.93|      593.98|2023-12-04|
|       5|          1|          1005|         3|       2|          13.74|      859.02|2023-12-04|
|       6|          5|          1006|         3|       2|           1.76|       871.0|2023-12-04|
|       7|          4|          1007|         1|       3|           4.31|      261.04|2023-12-03|
|       8|          

In [6]:
df.count()

78

In [ ]:
df_cleaned=df.dropDuplicates().dropna()

In [11]:
df_cleaned.show()

+--------+-----------+--------------+----------+--------+---------------+------------+----------+
|order_id|customer_id|transaction_id|product_id|quantity|discount_amount|total_amount|order_date|
+--------+-----------+--------------+----------+--------+---------------+------------+----------+
|       5|          1|          1005|         3|       2|          13.74|      859.02|2023-12-04|
|      16|          1|          1016|         5|       2|           6.81|      397.13|2023-12-03|
|      18|          1|          1018|         1|       3|          13.33|      252.02|2023-12-03|
|      19|          4|          1019|         5|       1|           3.89|      198.08|2023-12-01|
|      25|          2|          1025|         4|       3|          13.84|      357.89|2023-12-01|
|      26|          2|          1026|         2|       2|          13.09|      458.93|2023-12-02|
|      31|          2|          1031|         2|       1|           8.73|      227.28|2023-12-02|
|      32|          

In [14]:
from pyspark.sql.functions import sum

In [17]:
df_agg=df_cleaned.groupBy("customer_id").agg(sum("total_amount").alias("total_amount"))

In [18]:
df_agg.show()

+-----------+------------+
|customer_id|total_amount|
+-----------+------------+
|          4|     4933.92|
|          2|     8026.76|
|          5|     5906.82|
|          1|     9070.99|
|          3|     3690.87|
+-----------+------------+



In [ ]:
df.show()

+--------+-----------+--------------+----------+--------+---------------+------------+----------+
|order_id|customer_id|transaction_id|product_id|quantity|discount_amount|total_amount|order_date|
+--------+-----------+--------------+----------+--------+---------------+------------+----------+
|       1|          1|          1001|         3|       3|           9.02|     1300.12|2023-12-02|
|       2|          5|          1002|         4|       3|          17.74|      353.99|2023-12-04|
|       3|          1|          1003|         5|       2|          14.78|      389.16|2023-12-05|
|       4|          5|          1004|         5|       3|          11.93|      593.98|2023-12-04|
|       5|          1|          1005|         3|       2|          13.74|      859.02|2023-12-04|
|       6|          5|          1006|         3|       2|           1.76|       871.0|2023-12-04|
|       7|          4|          1007|         1|       3|           4.31|      261.04|2023-12-03|
|       8|          

In [26]:
df_cleaned.createOrReplaceTempView("sales")

In [ ]:
df_cleaned.write.mode("overwrite").format("delta").save("sales_delta")

In [27]:
spark.sql("select * from sales").show()

+--------+-----------+--------------+----------+--------+---------------+------------+----------+
|order_id|customer_id|transaction_id|product_id|quantity|discount_amount|total_amount|order_date|
+--------+-----------+--------------+----------+--------+---------------+------------+----------+
|       5|          1|          1005|         3|       2|          13.74|      859.02|2023-12-04|
|      16|          1|          1016|         5|       2|           6.81|      397.13|2023-12-03|
|      18|          1|          1018|         1|       3|          13.33|      252.02|2023-12-03|
|      19|          4|          1019|         5|       1|           3.89|      198.08|2023-12-01|
|      25|          2|          1025|         4|       3|          13.84|      357.89|2023-12-01|
|      26|          2|          1026|         2|       2|          13.09|      458.93|2023-12-02|
|      31|          2|          1031|         2|       1|           8.73|      227.28|2023-12-02|
|      32|          

In [24]:
spark.sql("select order_id, customer_id, product_id, quantity, order_date from sales").show()

+--------+-----------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|order_date|
+--------+-----------+----------+--------+----------+
|       1|          1|         3|       3|2023-12-02|
|       2|          5|         4|       3|2023-12-04|
|       3|          1|         5|       2|2023-12-05|
|       4|          5|         5|       3|2023-12-04|
|       5|          1|         3|       2|2023-12-04|
|       6|          5|         3|       2|2023-12-04|
|       7|          4|         1|       3|2023-12-03|
|       8|          3|         5|       4|2023-12-02|
|       9|          3|         5|       1|2023-12-02|
|      10|          1|         3|       2|2023-12-03|
|      11|          3|         4|       5|2023-12-02|
|      12|          2|         5|       4|2023-12-01|
|      13|          2|         1|       5|2023-12-01|
|      14|          3|         5|       2|2023-12-02|
|      15|          3|         2|       5|2023-12-05|
|      16|          1|      

In [29]:
spark.sql("select customer_id, round(sum(total_amount)) as total_amount from sales group by customer_id").show()

+-----------+------------+
|customer_id|total_amount|
+-----------+------------+
|          4|      4934.0|
|          2|      8027.0|
|          5|      5907.0|
|          1|      9071.0|
|          3|      3691.0|
+-----------+------------+



In [ ]:
df = spark.read.csv(input_path, header=True, inferSchema=True)
# Read is lazy; show() runs the job so you get output (tiny file = sub-second after Spark is up).
#df.show(5, truncate=False)

+--------+-----------+--------------+----------+--------+---------------+------------+----------+
|order_id|customer_id|transaction_id|product_id|quantity|discount_amount|total_amount|order_date|
+--------+-----------+--------------+----------+--------+---------------+------------+----------+
|1       |1          |1001          |3         |3       |9.02           |1300.12     |2023-12-02|
|2       |5          |1002          |4         |3       |17.74          |353.99      |2023-12-04|
|3       |1          |1003          |5         |2       |14.78          |389.16      |2023-12-05|
|4       |5          |1004          |5         |3       |11.93          |593.98      |2023-12-04|
|5       |1          |1005          |3         |2       |13.74          |859.02      |2023-12-04|
+--------+-----------+--------------+----------+--------+---------------+------------+----------+
only showing top 5 rows


In [ ]:
Medallion architecture


Bronze silver gold
raw data
refined data
business data



t1>>t2>>t3

